<a href="https://colab.research.google.com/github/Swayam-dot-cmd/neural-information-retrieval/blob/main/notebooks/01_bm25_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install beir rank_bm25


In [ ]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

out_dir = "./datasets"

data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

print("Corpus size:", len(corpus))
print("Queries:", len(queries))

In [ ]:
from rank_bm25 import BM25Okapi

# Extract document texts
doc_ids = list(corpus.keys())
corpus_texts = [corpus[doc_id]["text"] for doc_id in doc_ids]

# Tokenize
tokenized_corpus = [doc.split() for doc in corpus_texts]

# Build BM25 model
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready")

In [ ]:
import numpy as np

# Pick one query
query_id = list(queries.keys())[0]
query = queries[query_id]

tokenized_query = query.split()

# Get scores
scores = bm25.get_scores(tokenized_query)

# Top 5 docs
top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]

top_docs = [doc_ids[i] for i in top_indices]

print("Query:", query)
print("\nTop Documents:")
for doc_id in top_docs:
    print("-", corpus[doc_id]["text"][:200])

In [ ]:
def bm25_retrieve(query, top_k=10):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [doc_ids[i] for i in top_indices]

In [ ]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)
    return len(retrieved_k & relevant_set) / len(relevant_set)

In [ ]:
recalls = []

for qid in queries:
    query = queries[qid]

    # Retrieve docs
    retrieved_docs = bm25_retrieve(query, top_k=10)

    # Ground truth
    relevant_docs = list(qrels[qid].keys())

    # Compute recall
    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

# Final score
avg_recall = sum(recalls) / len(recalls)

print("BM25 Recall@10:", avg_recall)

In [ ]:
print("Final BM25 Recall@10:", avg_recall)